# 03 — Reinforcement Learning: RL Maze · RLHF · RL+RLHF composition

This notebook covers classic RL on a grid maze, PPO-based RLHF for language
model fine-tuning, and the combined RL+RLHF composition that bridges both.

## Architecture overview

```
  RL Maze
  ┌─────────────┐   state     ┌───────────┐   action   ┌───────────┐
  │  MazeEnv    │ ──────────▶ │  Q/DQN/   │ ──────────▶│  MazeEnv  │
  │  5×5..8×8   │ ◀────────── │  PPO agent│            │  (step)   │
  └─────────────┘   reward    └───────────┘            └───────────┘

  RLHF (PPO on Transformer)
  Stage 1: Pretrain LM on Shakespeare (CE loss)
  Stage 2: PPO fine-tune
    prompt → LM → response → reward(shakespearean_score)
    KL penalty vs frozen reference policy

  RL+RLHF composition
  Phase 1: DQN on maze
  Phase 2: encode trajectories as chars → pretrain Transformer LM
  Phase 3: RLHF — sample action sequences from LM, execute in maze,
            use binary maze reward (0/1) for PPO update
```

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_JUPYTER_TOKEN'))
if IN_COLAB:
    !pip install -e . -q

sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))
from viz import plot_metrics, show_maze
from mini_networks.colab.launcher import run_model

---
## 1. RL Maze — DQN agent

Trains a Deep Q-Network on a procedurally generated grid maze.
- **Replay buffer:** breaks temporal correlations for stable training
- **Target network:** fixed Q-targets prevent oscillation  
- **Epsilon-greedy:** exploration decays over episodes

In [ ]:
rl_logger = run_model('rl_maze', fast_demo=True)

In [ ]:
plot_metrics(rl_logger, keys=['episode_reward', 'success', 'epsilon'])

In [ ]:
# Visualise maze + greedy agent trajectory
from mini_networks.models.rl_maze.config import RLMazeConfig
from mini_networks.models.rl_maze.trainer import RLMazeTrainer, make_rl_maze_dataloader
from mini_networks.core.logging.logger import Logger
import tempfile

config = RLMazeConfig(fast_demo=True, agent_type='dqn')
trainer = RLMazeTrainer()

with tempfile.TemporaryDirectory() as tmp:
    logger = Logger(tmp, 'rl_demo')
    dl = make_rl_maze_dataloader(config)
    trainer.train(config, dl, logger)

# Show the maze layout
render = trainer.env.render()
show_maze(render, title='Maze layout')

# Run greedy episode
result = trainer.infer(config, {})
print(f'Actions: {result["actions"]}')
print(f'Steps: {result["steps"]}  Reward: {result["total_reward"]:.2f}  Success: {result["success"]}')

---
## 2. RLHF — PPO fine-tuning of TransformerLM

Two-stage pipeline:
1. Pretrain on Shakespeare with standard CE loss
2. PPO with Shakespearean vocabulary reward + KL penalty vs frozen reference policy

In [ ]:
rlhf_logger = run_model('rlhf', fast_demo=True)

In [ ]:
plot_metrics(rlhf_logger, keys=['pretrain_loss', 'avg_reward', 'ppo_loss'])

In [ ]:
from mini_networks.models.rlhf.config import RLHFConfig
from mini_networks.models.rlhf.trainer import RLHFTrainer

config = RLHFConfig(fast_demo=True)
trainer = RLHFTrainer()
trainer.load_checkpoint(config, rlhf_logger.artifacts_dir)

result = trainer.infer(config, {'prompt': 'Shall I compare thee', 'max_new_tokens': 60})
from viz import show_text_sample
show_text_sample(result['generated'], title=f'RLHF generation (reward={result["reward"]:.3f})')

---
## 3. RL + RLHF Maze composition

The full three-phase pipeline:
1. **DQN** learns to navigate the maze
2. **LM pretrain** on DQN trajectory corpus (`S→actions→G/X`)
3. **RLHF** fine-tune LM with maze binary reward

The `compare()` output shows DQN success rate vs LM success rate.

In [ ]:
from mini_networks.compositions.rl_rlhf_maze import RLHFMazeConfig, RLHFMazeComposition
from mini_networks.core.logging.logger import Logger
import tempfile

config = RLHFMazeConfig(fast_demo=True)
comp   = RLHFMazeComposition()

with tempfile.TemporaryDirectory() as tmp:
    logger = Logger(tmp, 'rl_rlhf_demo')
    comp.train(config, logger)
    plot_metrics(logger, keys=['episode_reward', 'pretrain_loss', 'avg_reward'])

result = comp.compare(config)
result

In [ ]:
# Print comparison table
print(f"{'Metric':<25} {'DQN':>10} {'LM (RLHF)':>12}")
print('-' * 50)
print(f"{'Success rate':<25} {result['dqn_success_rate']:>10.1%} {result['lm_success_rate']:>12.1%}")
print(f"{'Mean steps':<25} {result['dqn_mean_steps']:>10.1f} {result['lm_mean_steps']:>12.1f}")
print()
print('Sample LM trajectories:')
for t in result['sample_trajectories']:
    print(f'  {t}')